[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_B.ipynb)


# Set B — Cellular Models (with Electrophysiology Basics Appendix)

**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

Run the **Starter** once, then B1–B4 (models) and the **B‑Appendix** (Electrophysiology basics).

---
## Table of Contents
- [🔰 B0 Starter](#b0)
- [B1 LIF](#b1)
- [B2 Izhikevich](#b2)
- [B3 (optional) HH via NEURON](#b3)
- [B4 micro:bit → current clamp (CSV)](#b4)
- [B‑Appendix: Electrophysiology Basics](#bapp)
  - [B‑A.1 What an intracellular electrode measures](#ba1)
  - [B‑A.2 Current clamp fundamentals](#ba2)
  - [B‑A.3 Voltage clamp fundamentals](#ba3)
  - [B‑A.4 Estimate R_in and τ](#ba4)
  - [B‑A.5 I–V curve & E_rev](#ba5)
  - [B‑A.6 Threshold, rheobase, refractory](#ba6)
- [Reflection](#reflection)
---


### 🔰 B0 Starter (run once) <a id='b0'></a>

In [ ]:

# Lightweight install
!pip -q install numpy matplotlib --upgrade >/dev/null 2>&1

# Try NEURON if available (for B3 and Appendix cells)
try:
    import google
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    if IN_COLAB:
        import neuron
    else:
        import neuron
    HAVE_NEURON = True
except Exception:
    # attempt install in Colab only
    HAVE_NEURON = False
    if IN_COLAB:
        try:
            !pip -q install neuron==8.2.4 >/dev/null 2>&1
            import neuron
            HAVE_NEURON = True
        except Exception:
            HAVE_NEURON = False

import numpy as np
import matplotlib.pyplot as plt
print({'IN_COLAB':IN_COLAB, 'HAVE_NEURON':HAVE_NEURON})


## B1 — LIF <a id='b1'></a>

In [ ]:

import numpy as np, matplotlib.pyplot as plt
V_rest=-65; V_th=-50; V_reset=-70; R=100.0; tau=20e-3
T=1.0; dt=1e-4; t=np.arange(0,T,dt)
I=np.zeros_like(t); I[(t>0.1)&(t<0.6)]=0.5; I[(t>0.7)&(t<0.8)]=0.8
V=np.ones_like(t)*V_rest; spikes=[]
for k in range(1,len(t)):
    dV = (-(V[k-1]-V_rest) + R*I[k-1])*(dt/tau)
    V[k] = V[k-1] + dV
    if V[k]>=V_th:
        spikes.append(t[k]); V[k]=V_reset
plt.figure(figsize=(7,3.3))
plt.plot(t,V,'k'); plt.ylabel('V (mV)'); plt.title('B1: LIF')
plt.twinx(); plt.plot(t,I,'C0',alpha=0.5); plt.ylabel('I (nA)')
plt.xlabel('Time (s)'); plt.tight_layout(); plt.show()
print('Spikes:', len(spikes))


## B2 — Izhikevich <a id='b2'></a>

In [ ]:

import numpy as np, matplotlib.pyplot as plt
# Regular spiking parameters
a,b,c,d = 0.02,0.2,-65,8
T=1000; I=np.zeros(T); I[200:800]=10
v=-65*np.ones(T); u=b*v
for k in range(1,T):
    dv = 0.04*v[k-1]**2 + 5*v[k-1] + 140 - u[k-1] + I[k-1]
    du = a*(b*v[k-1]-u[k-1])
    v[k] = v[k-1] + 0.5*dv; u[k] = u[k-1] + 0.5*du
    v[k] = v[k] + 0.5*dv;   u[k] = u[k] + 0.5*du
    if v[k]>=30:
        v[k-1]=30; v[k]=c; u[k]+=d
plt.figure(figsize=(7,3.3)); plt.plot(v,'k')
plt.title('B2: Izhikevich (RS)'); plt.xlabel('Steps'); plt.ylabel('v (mV)'); plt.tight_layout(); plt.show()


## B3 — (optional) HH via NEURON <a id='b3'></a>

In [ ]:

try:
    from neuron import h
    soma=h.Section(name='soma'); soma.L=soma.diam=12.6157; soma.insert('hh')
    stim=h.IClamp(soma(0.5)); stim.delay=100; stim.dur=500; stim.amp=0.1
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65); h.continuerun(800)
    import numpy as np, matplotlib.pyplot as plt
    plt.figure(figsize=(7,3.3)); plt.plot(np.array(t),np.array(v))
    plt.title('B3: HH current clamp'); plt.xlabel('ms'); plt.ylabel('mV'); plt.tight_layout(); plt.show()
except Exception as e:
    print('NEURON not available; skip HH demo')


## B4 — micro:bit → current clamp (CSV pathway) <a id='b4'></a>

In [ ]:

import io, csv, numpy as np, matplotlib.pyplot as plt
from google.colab import files
uploaded = files.upload(); name=list(uploaded.keys())[0]
raw=uploaded[name].decode('utf-8'); rows=[r for r in csv.reader(io.StringIO(raw)) if len(r)>0]
try:
    _=[float(x) for x in rows[0]]; header=None
except Exception:
    header=rows[0]; rows=rows[1:]
import numpy as np
A=np.array([[float(x) for x in r[:2]] if len(r)>=2 else [np.nan,float(r[0])] for r in rows])
if np.isnan(A[:,0]).all():
    dt_ms=10.0; t_ms=np.arange(len(A))*dt_ms; accel=A[:,1]
else:
    t_ms=A[:,0]; accel=A[:,1]
baseline=np.percentile(accel,10); k=0.02
I = k*(accel-baseline); I[I<0]=0
plt.figure(figsize=(7,3)); plt.plot(t_ms,accel,'gray'); plt.title('accel (a.u.)'); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,3)); plt.plot(t_ms,I,'C0'); plt.title('mapped I (nA)'); plt.tight_layout(); plt.show()


---
## B‑Appendix: Electrophysiology Basics <a id='bapp'></a>
Concise essentials used across later sets. The following cells require **NEURON**.

### B‑A.1 What an intracellular electrode measures <a id='ba1'></a>

In [ ]:

try:
    from neuron import h
    import numpy as np, matplotlib.pyplot as plt
    t=[]
    for v0 in [-75,-65,-55]:
        h.finitialize(v0); h.continuerun(40)
        t.append(v0)
    print('Vm baseline depends on leak reversal & setup; this cell is a placeholder context note.')
except Exception:
    print('Requires NEURON: concept note only in this environment')


### B‑A.2 Current clamp fundamentals <a id='ba2'></a>

In [ ]:

try:
    from neuron import h
    soma=h.Section(name='somaB'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('hh'); soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65
    stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=50
    import numpy as np, matplotlib.pyplot as plt
    for A in [-0.05,0.05,0.10,0.15]:
        stim.amp=A
        v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
        h.finitialize(-65); h.continuerun(70)
        plt.figure(figsize=(7,3)); plt.plot(t,v,'k'); plt.title(f'B‑A.2: I‑clamp amp={A} nA'); plt.xlabel('ms'); plt.ylabel('mV'); plt.tight_layout(); plt.show()
except Exception:
    print('NEURON not available; skipping plots')


### B‑A.3 Voltage clamp fundamentals <a id='ba3'></a>

In [ ]:

try:
    from neuron import h
    soma=h.Section(name='somaC'); soma.L=soma.diam=20; soma.insert('hh');
    vcl=h.VClamp(soma(0.5)); vcl.dur[0]=10; vcl.amp[0]=-80; vcl.dur[1]=20; vcl.amp[1]=-20; vcl.dur[2]=20; vcl.amp[2]=40
    i=h.Vector().record(vcl._ref_i); v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65); h.continuerun(50)
    import numpy as np, matplotlib.pyplot as plt
    plt.figure(figsize=(7,4));
    plt.subplot(2,1,1); plt.plot(t,v,'k'); plt.ylabel('V (mV)'); plt.title('B‑A.3 Voltage clamp protocol')
    plt.subplot(2,1,2); plt.plot(t,i,'C1'); plt.xlabel('ms'); plt.ylabel('I (nA)'); plt.tight_layout(); plt.show()
except Exception:
    print('NEURON not available; skipping VC')


### B‑A.4 Estimate R_in and τ <a id='ba4'></a>

In [ ]:

try:
    from neuron import h
    soma=h.Section(name='somaR'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65
    stim=h.IClamp(soma(0.5)); stim.amp=-0.03; stim.delay=5; stim.dur=30
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65); h.continuerun(50)
    import numpy as np
    tnp=np.array(t); vnp=np.array(v)
    ss=(tnp>33)&(tnp<38); dV=vnp[ss].mean()-(-65)
    Rin_MOhm=abs(dV)/abs(stim.amp)
    v0=-65; vss=-65+dV; target=v0+0.632*(vss-v0)
    idx=np.where(vnp>=target)[0]
    tau_est=(tnp[idx[0]]-5) if len(idx)>0 else float('nan')
    print(f'R_in ≈ {Rin_MOhm:.1f} MΩ; τ ≈ {tau_est:.2f} ms')
except Exception:
    print('NEURON not available')


### B‑A.5 I–V curve & E_rev <a id='ba5'></a>

In [ ]:

try:
    from neuron import h
    import numpy as np, matplotlib.pyplot as plt
    soma=h.Section(name='somaIV'); soma.L=soma.diam=20; soma.insert('hh')
    steps=[-90,-70,-50,-30,-10,10,30,50]; I_ss=[]
    for Vc in steps:
        vcl=h.VClamp(soma(0.5)); vcl.dur[0]=5; vcl.amp[0]=-80; vcl.dur[1]=30; vcl.amp[1]=Vc; vcl.dur[2]=5; vcl.amp[2]=-80
        i=h.Vector().record(vcl._ref_i); t=h.Vector().record(h._ref_t)
        h.finitialize(-65); h.continuerun(40)
        tnp=np.array(t); inp=np.array(i)
        idx=(tnp>=30)&(tnp<=34); I_ss.append(inp[idx].mean())
    plt.figure(figsize=(5,3.8)); plt.plot(steps,I_ss,'o-'); plt.axhline(0,color='k',lw=1)
    plt.xlabel('Cmd V (mV)'); plt.ylabel('Steady I (nA)'); plt.title('B‑A.5 I–V'); plt.tight_layout(); plt.show()
except Exception:
    print('NEURON not available')


### B‑A.6 Threshold, rheobase, refractory <a id='ba6'></a>

In [ ]:

try:
    from neuron import h
    import numpy as np, matplotlib.pyplot as plt
    soma=h.Section(name='somaRheo'); soma.L=soma.diam=20; soma.insert('hh')
    amps=[0.02,0.04,0.06,0.08,0.10,0.12]
    stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=100
    for A in amps:
        stim.amp=A; v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
        h.finitialize(-65); h.continuerun(120)
        plt.figure(figsize=(6,2.3)); plt.plot(t,v,'k'); plt.title(f'B‑A.6: Staircase A={A} nA'); plt.tight_layout(); plt.show()
    # paired pulses
    stim1=h.IClamp(soma(0.5)); stim1.dur=2; stim1.amp=0.2
    stim2=h.IClamp(soma(0.5)); stim2.dur=2; stim2.amp=0.2
    for isi in [2,4,6,8,10,12]:
        stim1.delay=5; stim2.delay=5+isi
        v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
        h.finitialize(-65); h.continuerun(25)
        plt.figure(figsize=(6,2.3)); plt.plot(t,v,'k'); plt.title(f'B‑A.6: ISI={isi} ms'); plt.tight_layout(); plt.show()
except Exception:
    print('NEURON not available')


## Reflection <a id='reflection'></a>
- Which model (LIF vs Izhikevich) best matches your B‑Appendix measures?
- How do τ and R_in expectations from B‑Appendix show up in the models’ step responses?


## Practice / Discussion Questions — Set B — Biology → Model Mapping

1) Define a **model abstraction** for a neuron that preserves interpretability: which parameters must be in **biophysical units**, and why?
2) Explain how **blocks** (RC membrane, synaptic conductance, spike generator) can be composed into a testable single‑cell model.
3) *Justify*: What’s gained and lost when moving from detailed HH‑type to reduced spiking models?
4) How do you **document assumptions** so a reader can reproduce and critique your model? (List 3–4 items.)
5) What’s a **sanity check** for a synapse implemented with a fixed reversal potential? (Describe the test and expected result.)
6) Outline a minimal **reproducible workflow** (notebook → plots → metrics) for a membrane step test.
7) Why is **parameter transparency** essential for educational modeling? Give one consequence of opaque scaling.
8) *Design*: Write 3 summary metrics you would log for “EPSP → spike coupling” experiments.
9) Choose one **modeling decision** (e.g., omit Ca²⁺) and argue when it’s acceptable for instruction.
10) What is the **role of NEURON** in enabling reproducibility for single cells and small networks?
11) *Compare*: When do you favor **current clamp** vs **conductance injection** to test a component?
12) Define a **minimal dendrite** you would include to capture location effects without overcomplicating the model.
13) What’s the educational **trade‑off** of adding noise sources to a beginner model?
14) How would you **validate** your inhibitory synapse timing motif before using it in a network?
15) *Scenario*: Your EPSPs look too broad. List 3 plausible causes and the diagnostic you’d run.
16) Why do we advocate **fixed random seeds** in notebooks for teaching?
17) Provide a **short rubric** for students to self‑assess whether their model description is reproducible by others.
18) Suggest two **figures** (plots/tables) that best communicate a membrane and synapse validation.
19) Give an example of a **missing mechanism** signaled by a consistent mismatch between model and data.
20) *Reflect*: What will you carry from Set B into Set C–E when you begin analyzing spatial effects?
